In [1]:
import pickle
import pandas as pd
import numpy as np

from pulp import LpProblem, LpVariable, LpMinimize, lpSum, LpBinary, PULP_CBC_CMD, LpStatus

from src.data_loader import load_data

In [2]:
MAX_TASKS_PER_WORKER_PER_DAY = 3

In [3]:
# Load pipeline to forecast task duration
MODEL_PATH = 'src/task2/duration_fcst_pipe.pkl'

with open(MODEL_PATH, 'rb') as f:
    pipe = pickle.load(f)

In [4]:
# Load data and use the last date for schedule optimization
data = load_data('data_2')
data['date'] = data.time_from.dt.normalize()
data = data[data.date == data.date.max()]

# Generate workers list
workers = sorted(data['worker_id'].unique())
n_workers = len(workers)

# Randomly get 15 tasks (5 workers x 3 task per worker max)
max_tasks_per_day = MAX_TASKS_PER_WORKER_PER_DAY * n_workers
data = data.sample(max_tasks_per_day)
data

,log_id,worker_id,process_name,item_id,time_from,time_to,date
542,543,worker_1,placement,item_4,2025-09-30 16:22:22,2025-09-30 16:23:02.387,2025-09-30
3094,3095,worker_4,transfer,item_3,2025-09-30 13:39:24,2025-09-30 13:40:19.612,2025-09-30
1353,1354,worker_2,transfer,item_5,2025-09-30 22:40:34,2025-09-30 22:41:10.438,2025-09-30
3100,3101,worker_4,collection,item_1,2025-09-30 18:39:52,2025-09-30 18:40:26.010,2025-09-30
635,636,worker_1,collection,item_1,2025-09-30 20:05:21,2025-09-30 20:06:02.046,2025-09-30
624,625,worker_1,placement,item_5,2025-09-30 14:00:27,2025-09-30 14:01:16.031,2025-09-30
760,761,worker_1,placement,item_1,2025-09-30 14:22:07,2025-09-30 14:22:54.325,2025-09-30
1409,1410,worker_2,collection,item_2,2025-09-30 13:48:53,2025-09-30 13:49:13.267,2025-09-30
3902,3903,worker_5,collection,item_4,2025-09-30 12:27:41,2025-09-30 12:28:17.733,2025-09-30
2304,2305,worker_3,transfer,item_5,2025-09-30 14:01:04,2025-09-30 14:01:52.952,2025-09-30


In [5]:
# Generate tasks lists
tasks = sorted(data['log_id'].unique())

# Calculate time spent on all 15 tasks factually
data['duration'] = (data.time_to - data.time_from).dt.total_seconds()
initial_duration = sum(data.duration)
print(f'Initial total work duration {initial_duration} seconds')

data.drop(columns=['worker_id', 'time_to', 'date', 'duration'], inplace=True)

data = data.merge(
    pd.DataFrame({'worker_id': workers}),
    how='cross'
)

# Forecast duration:
data['predicted_duration'] = pipe.predict(data)
data['time_to_pred'] = data['time_from'] + pd.to_timedelta(data['predicted_duration'], unit='s')

data

Initial total work duration 633.349 seconds


,log_id,process_name,item_id,time_from,worker_id,predicted_duration,time_to_pred
0,543,placement,item_4,2025-09-30 16:22:22,worker_1,33.646229,2025-09-30 16:22:55.646228694
1,543,placement,item_4,2025-09-30 16:22:22,worker_2,30.727038,2025-09-30 16:22:52.727037906
2,543,placement,item_4,2025-09-30 16:22:22,worker_3,35.335957,2025-09-30 16:22:57.335956753
3,543,placement,item_4,2025-09-30 16:22:22,worker_4,34.311392,2025-09-30 16:22:56.311392146
4,543,placement,item_4,2025-09-30 16:22:22,worker_5,49.749693,2025-09-30 16:23:11.749693326
...,...,...,...,...,...,...,...
70,3173,placement,item_1,2025-09-30 15:10:53,worker_1,31.919037,2025-09-30 15:11:24.919037458
71,3173,placement,item_1,2025-09-30 15:10:53,worker_2,29.164198,2025-09-30 15:11:22.164197707
72,3173,placement,item_1,2025-09-30 15:10:53,worker_3,33.631444,2025-09-30 15:11:26.631444346
73,3173,placement,item_1,2025-09-30 15:10:53,worker_4,32.626783,2025-09-30 15:11:25.626783337


In [6]:
# Cost dict
cost = data.set_index(['worker_id', 'log_id'])['predicted_duration'].to_dict()

# Process dict
process = data.drop_duplicates('log_id')\
    .set_index('log_id')['process_name']\
    .to_dict()

# Optimization model
model = LpProblem('WorkerAssignment', LpMinimize)

# Decision variables
x = LpVariable.dicts(
    'assign',
    ((w, t) for w in workers for t in tasks),
    cat=LpBinary
)

# Objective function
model += lpSum(cost[(w, t)] * x[(w, t)] for w in workers for t in tasks)

In [7]:
# Constraint 1: every task assigned exactly once
for t in tasks:
    model += (lpSum(x[(w, t)] for w in workers) == 1, f'Assign_{t}')

# Constraint 2: max tasks per worker
for w in workers:
    model += (lpSum(x[(w, t)] for t in tasks) <= MAX_TASKS_PER_WORKER_PER_DAY, f'MaxTasks_{w}')

# Constraint 3: max 1 simultaneous tasks
time_points = sorted(pd.concat([data['time_from'], data['time_to_pred']]).unique())

for w in workers:
    worker_df = data[data.worker_id == w].set_index('log_id')
    
    for tp in time_points:
        active_tasks = worker_df[
            (worker_df.time_from <= tp) &
            (worker_df.time_to_pred > tp)
        ].index.tolist()

        if len(active_tasks) > 1:
            model += (lpSum(x[(w, t)] for t in active_tasks) <= 1, f'Overlap_{w}_{tp}')
            
# Constraint 4: worker_2 cannot perform placement
for t in tasks:
    if process[t] == 'placement':
        model += (x[('worker_2', t)] == 0, f'NoPlacement_worker2_{t}')

In [8]:
# Solve:
solver = PULP_CBC_CMD(msg=True)
status = model.solve(solver)
print(f'Status: {LpStatus[status]}')

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /workspaces/optimization/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/d85179d884dc403082309ad577a154cb-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /tmp/d85179d884dc403082309ad577a154cb-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 32 COLUMNS
At line 416 RHS
At line 444 BOUNDS
At line 520 ENDATA
Problem MODEL has 27 rows, 75 columns and 158 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 568.594 - 0.00 seconds
Cgl0002I 6 variables fixed
Cgl0004I processed model has 21 rows, 69 columns (69 integer (69 of which binary)) and 140 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of 568.594
Cbc0038I Before mini branch and bound, 69 integers at bound fixed and 0 continuous
Cbc0038I Mini branch and bou

In [9]:
# Show results
result = []

for w in workers:
    for t in tasks:
        if x[(w, t)].value() == 1:
            row = data[(data.worker_id == w) & (data.log_id == t)].iloc[0]
            result.append({
                'log_id': t,
                'worker_id': w,
                'process_name': row.process_name,
                'time_from': row.time_from,
                'time_to_pred': row.time_to_pred,
                'predicted_duration': row.predicted_duration
            })

assignment = pd.DataFrame(result).sort_values('time_from')
assignment

,log_id,worker_id,process_name,time_from,time_to_pred,predicted_duration
11,2321,worker_4,placement,2025-09-30 11:01:08,2025-09-30 11:01:42.311392146,34.311392
5,3903,worker_2,collection,2025-09-30 12:27:41,2025-09-30 12:28:07.850078505,26.850079
14,3976,worker_5,transfer,2025-09-30 13:20:58,2025-09-30 13:21:56.485452295,58.485452
13,3095,worker_5,transfer,2025-09-30 13:39:24,2025-09-30 13:40:22.485452295,58.485452
7,1410,worker_3,collection,2025-09-30 13:48:53,2025-09-30 13:49:22.122423320,29.122423
0,625,worker_1,placement,2025-09-30 14:00:27,2025-09-30 14:01:01.787712485,34.787712
4,2305,worker_2,transfer,2025-09-30 14:01:04,2025-09-30 14:01:43.898538319,39.898538
10,761,worker_4,placement,2025-09-30 14:22:07,2025-09-30 14:22:39.626783337,32.626783
2,3173,worker_1,placement,2025-09-30 15:10:53,2025-09-30 15:11:24.919037458,31.919037
9,543,worker_4,placement,2025-09-30 16:22:22,2025-09-30 16:22:56.311392146,34.311392


In [10]:
final_duration = sum(assignment.predicted_duration)
print(f'Tasks duration change after the reassignment: {np.round((final_duration - initial_duration) / initial_duration * 100, 2)}%')

Tasks duration change after the reassignment: -10.22%


In [11]:
# Statistics by worker:
assignment.groupby('worker_id').agg(
    n_tasks=('log_id', 'count'), 
    total_duration=('predicted_duration', 'sum')
)

,n_tasks,total_duration
worker_id,,
worker_1,3,105.169437
worker_2,3,106.647155
worker_3,3,86.828462
worker_4,3,101.249568
worker_5,3,168.698910
